# 🎭 Wav2Lip — Lip Sync Batch Pipeline (Colab Edition)
### Novella Love Tales
**Flow:** S3 audio → Wav2Lip → S3 output

**One-time setup:**
1. Runtime → Change runtime type → Hardware accelerator → **GPU (T4)**
2. Click the 🔑 key icon in the left sidebar → add secrets: `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`, `AWS_REGION`, `HF_TOKEN` (optional) → toggle **Notebook access** on for each
3. Run cells top to bottom. The main loop (last cell) is the one to re-run each session.

**Speed features:**
- ✅ S3 skip — files already in output folder are never re-processed
- ✅ Output key set fetched once (no per-file S3 HEAD calls)
- ✅ 64 KB download chunks for faster model download
- ✅ Immediate disk cleanup after each file

In [1]:
# ============================================================
# CELL 1 — CONFIGURATION  (edit only this cell)
# ============================================================
import os
from google.colab import userdata

def get_secret(name, required=True):
    try:
        return userdata.get(name)
    except Exception:
        if required:
            raise RuntimeError(
                f"Secret '{name}' not found. Add it via the 🔑 key icon in the "
                f"left sidebar, then enable 'Notebook access' for it."
            )
        return None

os.environ["AWS_ACCESS_KEY_ID"]     = get_secret("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"] = get_secret("AWS_SECRET_ACCESS_KEY")
os.environ["AWS_DEFAULT_REGION"]    = get_secret("AWS_REGION")
os.environ["HF_TOKEN"]              = get_secret("HF_TOKEN", required=False) or ""

AWS_ACCESS_KEY = os.environ["AWS_ACCESS_KEY_ID"]
AWS_SECRET_KEY = os.environ["AWS_SECRET_ACCESS_KEY"]
AWS_REGION     = os.environ["AWS_DEFAULT_REGION"]

print("AWS credentials loaded successfully.")

S3_BUCKET             = 'mynovels-iqranaeem-247601'
ANIME_GIRL_IMAGE_KEY  = 'Next_YT/st1.png'  # S3 key, not a URI
RESIZE_FACTOR         = 2
PADS                  = [0, 10, 0, 0]
MAX_FILES_PER_RUN     = 20000
WORK_DIR              = '/content/wav2lip_work'
FPS = 15

# List of (input audio prefix, output lip prefix) pairs

AWS credentials loaded successfully.


In [2]:
FOLDER_PAIRS = [
    ("New_Novel/02/audio/", "New_Novel/02/lip/"),
    ("New_Novel/03/audio/", "New_Novel/03/lip/"),
    ("New_Novel/04/audio/", "New_Novel/04/lip/"),
    ("New_Novel/05/audio/", "New_Novel/05/lip/"),
    ("New_Novel/06/audio/", "New_Novel/06/lip/"),
    ("New_Novel/07/audio/", "New_Novel/07/lip/"),
    ("New_Novel/08/audio/", "New_Novel/08/lip/"),
    ("New_Novel/09/audio/", "New_Novel/09/lip/"),
    ("New_Novel/10/audio/", "New_Novel/10/lip/"),
    ("New_Novel/11/audio/", "New_Novel/11/lip/"),
    ("New_Novel/12/audio/", "New_Novel/12/lip/"),
    ("New_Novel/13/audio/", "New_Novel/13/lip/"),
]

In [3]:
# ============================================================
# CELL 2 — INSTALL & CLONE  (run once per session)
# ============================================================
import os, shutil, subprocess

os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)

if os.path.exists('Wav2Lip'):
    shutil.rmtree('Wav2Lip')

print('📦 Cloning Wav2Lip...')
env = os.environ.copy()
env['GIT_TERMINAL_PROMPT'] = '0'
subprocess.run(['git', 'clone', 'https://github.com/Rudrabha/Wav2Lip.git'], env=env, check=True)
os.chdir('Wav2Lip')

!pip install -q boto3 opencv-python soundfile pydub tqdm librosa==0.9.2

📦 Cloning Wav2Lip...


In [4]:
# ============================================================
# CELL 3 — DOWNLOAD MODEL (non-GAN, lower VRAM / faster)
# ============================================================
import requests, os
from tqdm import tqdm

MODEL_PATH = 'checkpoints/wav2lip.pth'   # <-- non-GAN checkpoint
os.makedirs('checkpoints', exist_ok=True)

if os.path.exists(MODEL_PATH) and os.path.getsize(MODEL_PATH) > 100_000_000:
    print('✅ Model already present — skipping download.')
else:
    if os.path.exists(MODEL_PATH):
        os.remove(MODEL_PATH)
    print('📥 Downloading Wav2Lip (non-GAN, ~145 MB)...')

    url = 'https://huggingface.co/Nekochu/Wav2Lip/resolve/main/wav2lip.pth'

    r = requests.get(url, stream=True)
    r.raise_for_status()
    total = int(r.headers.get('content-length', 0))
    with open(MODEL_PATH, 'wb') as f, tqdm(total=total, unit='B', unit_scale=True) as pbar:
        for chunk in r.iter_content(chunk_size=65536):
            f.write(chunk)
            pbar.update(len(chunk))
    print(f'✅ Model saved ({os.path.getsize(MODEL_PATH)/1e6:.1f} MB).')

# PyTorch 2.6 patch
import torch
_orig_load = torch.load
def _safe_load(*a, **kw):
    kw['weights_only'] = False
    return _orig_load(*a, **kw)
torch.load = _safe_load
print('✅ PyTorch 2.6 patch applied.')

📥 Downloading Wav2Lip (non-GAN, ~145 MB)...


100%|██████████| 436M/436M [00:01<00:00, 230MB/s]


✅ Model saved (435.8 MB).
✅ PyTorch 2.6 patch applied.


In [5]:
# ============================================================
# CELL 4 — HELPERS
# ============================================================
import boto3, cv2, librosa, time, subprocess, os
from botocore.client import Config

def _s3():
    return boto3.client(
        's3', region_name=AWS_REGION,
        aws_access_key_id=AWS_ACCESS_KEY,
        aws_secret_access_key=AWS_SECRET_KEY,
        config=Config(signature_version='s3v4')
    )

def fetch_done_keys(bucket, prefix):
    """Returns set of S3 keys already in output prefix. Called ONCE."""
    done = set()
    paginator = _s3().get_paginator('list_objects_v2')
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get('Contents', []):
            done.add(obj['Key'])
    return done

def list_audio_tasks(bucket, audio_prefix, output_prefix):
    tasks = []
    paginator = _s3().get_paginator('list_objects_v2')
    for page in paginator.paginate(Bucket=bucket, Prefix=audio_prefix):
        for obj in page.get('Contents', []):
            key = obj['Key']
            if key.lower().endswith('.wav'):
                out_key = key.replace(audio_prefix, output_prefix, 1).replace('.wav', '.mp4')
                tasks.append((key, out_key))
    return tasks

def s3_download(key, local):
    try:
        _s3().download_file(S3_BUCKET, key, local); return True
    except Exception as e:
        print(f'  ❌ Download failed: {e}'); return False

def s3_upload(local, key):
    try:
        _s3().upload_file(local, S3_BUCKET, key)
        print(f'  ✅ Uploaded → s3://{S3_BUCKET}/{key}'); return True
    except Exception as e:
        print(f'  ❌ Upload failed: {e}'); return False

def make_static_video(img_path, wav_path, out_path):
    duration = librosa.get_duration(filename=wav_path)
    img = cv2.imread(img_path)
    if img is None:
        raise FileNotFoundError(f'Image not found: {img_path}')
    if RESIZE_FACTOR > 1:
        img = cv2.resize(img, (img.shape[1]//RESIZE_FACTOR, img.shape[0]//RESIZE_FACTOR))
    h, w = img.shape[:2]
    out = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*'mp4v'), FPS, (w, h))
    for _ in range(int(duration * FPS)):
        out.write(img)
    out.release()
    return duration

def run_wav2lip(face_video, wav, outfile):
    cmd = [
        'python', 'inference.py',
        '--checkpoint_path', MODEL_PATH,
        '--face', face_video,
        '--audio', wav,
        '--outfile', outfile,
        '--resize_factor', str(RESIZE_FACTOR),
        '--pads', *map(str, PADS),
        '--crop', '0', '-1', '0', '-1',
        '--face_det_batch_size', '16',
        '--wav2lip_batch_size', '128',
        '--nosmooth'
    ]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print('  ❌ Wav2Lip stderr:', r.stderr[-500:]); return False
    return True

def cleanup(*paths):
    for p in paths:
        try: os.remove(p)
        except: pass

# verify S3
try:
    _s3().head_bucket(Bucket=S3_BUCKET)
    print('✅ S3 connected.')
except Exception as e:
    raise RuntimeError(f'S3 connection failed: {e}')

# download face image once (cv2.imread cannot read s3:// URIs)
LOCAL_FACE_IMAGE = f'{WORK_DIR}/st1.png'
if not s3_download(ANIME_GIRL_IMAGE_KEY, LOCAL_FACE_IMAGE):
    raise RuntimeError(f'Could not download face image: {ANIME_GIRL_IMAGE_KEY}')
print(f'✅ Face image downloaded → {LOCAL_FACE_IMAGE}')

✅ S3 connected.
✅ Face image downloaded → /content/wav2lip_work/st1.png


In [6]:
# ============================================================
# CELL 5 — MAIN LOOP  (re-run this cell each Colab session)
# ============================================================
total_done_overall = 0
files_processed_overall = 0

for S3_AUDIO_PREFIX, S3_OUTPUT_PREFIX in FOLDER_PAIRS:
    print(f'\n{"="*60}')
    print(f'📁 Folder: {S3_AUDIO_PREFIX} → {S3_OUTPUT_PREFIX}')
    print(f'{"="*60}')
    print('🔍 Scanning S3...')
    all_tasks  = list_audio_tasks(S3_BUCKET, S3_AUDIO_PREFIX, S3_OUTPUT_PREFIX)
    done_in_s3 = fetch_done_keys(S3_BUCKET, S3_OUTPUT_PREFIX)
    pending = [(ak, ok) for ak, ok in all_tasks if ok not in done_in_s3]
    n_skip  = len(all_tasks) - len(pending)
    print(f'📋 Total audio files : {len(all_tasks)}')
    print(f'   Already in S3     : {n_skip}  (skipping)')

    if not pending:
        print('🎉 All files already done in this folder!')
        continue

    remaining_budget = MAX_FILES_PER_RUN - files_processed_overall
    if remaining_budget <= 0:
        print('⏸️  Per-run file cap reached — stopping. Re-run Cell 5 to continue.')
        break

    batch = pending[:remaining_budget]
    print(f'   To process now    : {len(batch)}')

    for idx, (audio_key, out_key) in enumerate(batch, 1):
        label = os.path.basename(audio_key)
        print(f'\n[{idx}/{len(batch)}] {label}')
        wav  = f'{WORK_DIR}/in_{idx}.wav'
        face = f'{WORK_DIR}/face_{idx}.mp4'
        outp = f'{WORK_DIR}/out_{idx}.mp4'

        if not s3_download(audio_key, wav):
            cleanup(wav); continue

        try:
            dur = make_static_video(LOCAL_FACE_IMAGE, wav, face)
        except Exception as e:
            print(f'  ❌ Prep error: {e}'); cleanup(wav, face); continue

        t0 = time.time()
        ok = run_wav2lip(face, wav, outp)
        elapsed = time.time() - t0
        if not ok:
            cleanup(wav, face, outp); continue
        print(f'  ⏱️  {elapsed:.1f}s inference for {dur:.1f}s audio')

        if os.path.exists(outp) and os.path.getsize(outp) > 0:
            if s3_upload(outp, out_key):
                done_in_s3.add(out_key)
                total_done_overall += 1

        cleanup(wav, face, outp)
        files_processed_overall += 1

print(f'\n✅ Session done — {total_done_overall} files uploaded across all folders.')
print(f'🔄 {files_processed_overall} files processed this run (cap: {MAX_FILES_PER_RUN}). Re-run Cell 5 to continue with remaining files.')


📁 Folder: New_Novel/02/audio/ → New_Novel/02/lip/
🔍 Scanning S3...
📋 Total audio files : 946
   Already in S3     : 138  (skipping)
   To process now    : 808

[1/808] b_0139.wav


KeyboardInterrupt: 